<a href="https://colab.research.google.com/github/Pavlenko-Kateryna/da_course/blob/main/HW_15_4_%D0%90%D0%BD%D0%B0%D0%BB%D1%96%D0%B7_%D0%90_%D0%92_%D1%82%D0%B5%D1%81%D1%82%D1%96%D0%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

$$H_0: p = p_0$$
$$H_a: p \ne p_0$$

$p_0$ = 19

Маємо двосторонній тест

In [6]:
size = sms.proportion_effectsize(0.20,0.19)
size

np.float64(0.025241594409087353)

In [11]:
required_n = sms.NormalIndPower().solve_power(
    size,
    power=0.8,
    alpha=0.05,
    ratio=1
)

required_n = ceil(required_n)

required_n

24638

2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [40]:
df = pd.read_csv("/content/drive/MyDrive/cookie_cats.csv")
df

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True
...,...,...,...,...,...
90184,9999441,gate_40,97,True,False
90185,9999479,gate_40,30,False,False
90186,9999710,gate_30,28,True,False
90187,9999768,gate_40,51,True,False


In [10]:
df.groupby('version')['retention_7'].mean()

,retention_7
version,
gate_30,0.190201
gate_40,0.182000


Трохи краще утримання дає версія на 30 рівень гри

In [21]:
# контрольна група - 30 рівень, тестова - 40 рівень
control_sample = df[df['version'] == 'gate_30'].sample(n=required_n, random_state=22)
treatment_sample = df[df['version'] == 'gate_40'].sample(n=required_n, random_state=22)

ab_test = pd.concat([control_sample, treatment_sample], axis=0)
ab_test.reset_index(drop=True, inplace=True)
ab_test

,userid,version,sum_gamerounds,retention_1,retention_7
0,7540471,gate_30,45,True,False
1,3589138,gate_30,21,True,False
2,3177668,gate_30,14,True,False
3,2133884,gate_30,26,False,False
4,492763,gate_30,39,True,True
...,...,...,...,...,...
49271,1586632,gate_40,236,True,True
49272,7134057,gate_40,58,False,True
49273,7711092,gate_40,145,True,True
49274,207388,gate_40,6,False,False


In [41]:
ab_test['version'].value_counts()

,count
version,
gate_30,24638
gate_40,24638


3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


Так як маємо велику вибірку n > 30, у нашому випадку n = 24638, можемо використати z-тест для великої вибірки

In [42]:
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

In [43]:
control_results = ab_test[ab_test['version'] == 'gate_30']['retention_7']
treatment_results = ab_test[ab_test['version'] == 'gate_40']['retention_7']

In [44]:
n_control = control_results.count()
n_treatment = treatment_results.count()
successes = [control_results.sum(), treatment_results.sum()]
nobs = [n_control, n_treatment]

In [45]:
successes

[np.int64(4660), np.int64(4385)]

In [46]:
nobs

[np.int64(24638), np.int64(24638)]

In [48]:
z_stat, pval = proportions_ztest(successes,nobs=nobs)
(lower_con, lower_treat),(upper_con, upper_treat) = proportion_confint(successes,nobs=nobs,alpha=0.05)

print(f'z statistic: {z_stat}')
print(f'p-value: {pval}')
print(f'Довірчий інтервал 95% для групи control: [{lower_con}, {upper_con}]')
print(f'Довірчий інтервал 95% для групи treatment: [{lower_treat}, {upper_treat}]')

z statistic: 3.20011597372952
p-value: 0.0013737229952302259
Довірчий інтервал 95% для групи control: [0.1842487285867723, 0.19402872899907073]
Довірчий інтервал 95% для групи treatment: [0.17320105270117436, 0.18275316436189892]


Як бачимо за резульатами, що ми можемо відхилити нульову гіпотезу і конверсія не дорівнює 19%, але ми точно не знаємо, вона менше чи більше. Тому додатково проведемо правосторонній тест

$$H_0: p = 19$$
$$H_a: p > 19$$

In [56]:
z_stat, pval = proportions_ztest(successes,nobs=nobs, value = 0.19, alternative='larger')
(lower_con, lower_treat),(upper_con, upper_treat) = proportion_confint(successes,nobs=nobs,alpha=0.05)

print(f'z statistic: {z_stat}')
print(f'p-value: {pval}')
print(f'Довірчий інтервал 95% для групи control: [{lower_con}, {upper_con}]')
print(f'Довірчий інтервал 95% для групи treatment: [{lower_treat}, {upper_treat}]')

z statistic: -51.27423638460532
p-value: 1.0
Довірчий інтервал 95% для групи control: [0.1842487285867723, 0.19402872899907073]
Довірчий інтервал 95% для групи treatment: [0.17320105270117436, 0.18275316436189892]


p_value більше ніж 0.05, тому ми не можемо відхилити нульову гіпотезу і це означає, що конверсія не виросла, а навпаки впала, можемо перевірити це за допомогою лівосторонньому тесту нижче

$$H_0: p = 19$$
$$H_a: p < 19$$

In [58]:
z_stat, pval = proportions_ztest(successes,nobs=nobs, value = 0.19, alternative='smaller')
(lower_con, lower_treat),(upper_con, upper_treat) = proportion_confint(successes,nobs=nobs,alpha=0.05)

print(f'z statistic: {z_stat}')
print(f'p-value: {pval}')
print(f'Довірчий інтервал 95% для групи control: [{lower_con}, {upper_con}]')
print(f'Довірчий інтервал 95% для групи treatment: [{lower_treat}, {upper_treat}]')

z statistic: -51.27423638460532
p-value: 0.0
Довірчий інтервал 95% для групи control: [0.1842487285867723, 0.19402872899907073]
Довірчий інтервал 95% для групи treatment: [0.17320105270117436, 0.18275316436189892]


Так, p_value менше 0.05, тому відхиляємо нульову гіпотезу, і це означає, що конверсія зменшилась і менше ніж 19

Тому, можемо зробити висновок, що переміщення воріт на 10 рівнів пізніше негативно вплинуло на утримання.  Довірчий інтервал для treatment групи не включає навіть початкового відсотку утримання - 19% і не включає більше значення

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


Гіпотези:
$$H_0:$$
  версія гри і утримання на 7 день - незалежні

$$H_a:$$
  версія гри і утримання на 7 день - залежні

In [59]:
pd.crosstab(ab_test['version'], ab_test['retention_7'])

retention_7,False,True
version,,
gate_30,19978,4660
gate_40,20253,4385


In [60]:
crosstab = pd.crosstab(ab_test['version'], ab_test['retention_7'])

In [61]:
chi2, p, dof, expected = stats.chi2_contingency(crosstab)

print(f"χ² = {chi2}")
print(f"p-value = {p}")
print(f"Ступені свободи = {dof}")
print("Очікувані частоти:\n", expected)

χ² = 10.166399534671793
p-value = 0.0014302330491884595
Ступені свободи = 1
Очікувані частоти:
 [[20115.5  4522.5]
 [20115.5  4522.5]]


Отримали результат, що p_value < 0.05, тому ви відхиляємо нульову гіпотезу. Це означає, що версія гри і утримання на 7 день залежні.